# Fine-tuning Embeddings and Advanced Retrieval

## Overview

- Create a vector search endpoint

- Create a vector search index

- Ingest text data into the vector search index

- Query to retrieve documents

## Setups

In [0]:
%sql
-- setup to catalog and schema
use catalog `studies`;
use schema `databricks_finetuning`;

In [0]:
%sql
SHOW VOLUMES

database,volume_name
databricks_finetuning,article_infos


In [0]:
INPUT_DOCS_PATH = '/Volumes/studies/databricks_finetuning/article_infos'
VS_ENDPOINT_NAME='article_vs_endpoint'
VS_INDEX_NAME = 'article_vs_index'
CHUNKS_TABLE_NAME = 'article_chunks'

RUN_PARSE_STEP = False
RUN_CHUNK_STEP = False
CREATE_VS_INDEX = False

infos_env = spark.sql('SELECT current_catalog(), current_schema()').collect()[0]
CATALOG = infos_env[0]
SCHEMA = infos_env[1]
print(f'Catalog: {CATALOG}')
print(f'Schema: {SCHEMA}')

Catalog: studies
Schema: databricks_finetuning


In [0]:
spark.sql(f'list "{INPUT_DOCS_PATH}"').display()

path,name,size,modification_time
/Volumes/studies/databricks_finetuning/article_infos/pg2_abstract.pdf,pg2_abstract.pdf,136388,1771696177000
/Volumes/studies/databricks_finetuning/article_infos/pgs19_20_regulation.pdf,pgs19_20_regulation.pdf,244135,1771696177000
/Volumes/studies/databricks_finetuning/article_infos/pgs25_6_conclusion.pdf,pgs25_6_conclusion.pdf,52885,1771696177000
/Volumes/studies/databricks_finetuning/article_infos/pgs27_30_refs.pdf,pgs27_30_refs.pdf,81625,1771696177000
/Volumes/studies/databricks_finetuning/article_infos/pgs3_6_intro.pdf,pgs3_6_intro.pdf,140192,1771696177000
/Volumes/studies/databricks_finetuning/article_infos/pgs7_9_model.pdf,pgs7_9_model.pdf,267683,1771696177000


In [0]:
import json
from typing import List
from pyspark.sql.functions import expr

from databricks.sdk import WorkspaceClient
from databricks.vector_search.client import VectorSearchClient
from databricks.sdk.service.vectorsearch import EndpointType

import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import monotonically_increasing_id

In [0]:
ws_client = WorkspaceClient()
vs_client = VectorSearchClient(disable_notice=True)

## Create a vector search endpoint

In [0]:
def create_vs_endpoint(endpoint_name: str, ws_client: WorkspaceClient) -> None:
	'''Create an endpoint in the workspace.'''
	
	try:
		ws_client.vector_search_endpoints.get_endpoint(endpoint_name)
	except:
		ws_client.vector_search_endpoints.create_endpoint(
      		name=endpoint_name, 
        	endpoint_type=EndpointType('STANDARD')
			# Note: Maximum number of vector search endpoints per (student) workspace quota is 1. Compute > VEctor Search > Endpoints
       )

In [0]:
create_vs_endpoint(
    endpoint_name=VS_ENDPOINT_NAME, 
    ws_client=ws_client
)

## Parsing unstructured docs

In [0]:
if RUN_PARSE_STEP:
	parsed_docs_sql = spark.sql(f'''
	select
		path,
		ai_parse_document(
			content,
			map('version', '2.0')
		) as parsed_docs
	from read_files('{INPUT_DOCS_PATH}', format => 'binaryFile')
	''')
else:
    parsed_docs_sql = spark.read.table(f'{CATALOG}.{SCHEMA}.article_parsed_infos')
    
display(parsed_docs_sql)

path parsed_docs dbfs:/Volumes/studies/databricks_finetuning/article_infos/pg2_abstract.pdf {"document":{"elements":[{"bbox":[{"coord":[117,136,578,252],"page_id":0}],"content":"A Model of Online Misinformation\nDaron Acemoglu, Asuman Ozdaglar, and James Siderius\nNBER Working Paper No. 28884\nJune 2021, Revised January 2022\nJEL No. D83,D85,P16","description":null,"id":0,"type":"text"},{"bbox":[{"coord":[453,277,567,299],"page_id":0}],"content":"ABSTRACT","description":null,"id":1,"type":"section_header"},{"bbox":[{"coord":[118,323,907,626],"page_id":0}],"content":"We present a model of online content sharing where agents sequentially observe an article and must decide whether to share it with others. This content may or may not contain misinformation. Agents gain utility from positive social media interactions but do not want to be called out for propagating misinformation. We characterize the (Bayesian-Nash) equilibria of this social media game and show sharing exhibits strategic complementarity. Our first main result establishes that the impact of homophily on content virality is non-monotone: homophily reduces the broader circulation of an article, but it creates echo chambers that impose less discipline on the sharing of low-reliability content. This insight underpins our second main result, which demonstrates that social media platforms interested in maximizing engagement tend to design their algorithms to create more homophilic communication patterns (\"filter bubbles\"). We show that platform incentives to amplify misinformation are particularly pronounced for low-reliability content likely to contain misinformation and when there is greater polarization and more divisive content. Finally, we discuss various regulatory solutions to such platform-manufactured misinformation.","description":null,"id":2,"type":"text"},{"bbox":[{"coord":[117,673,430,834],"page_id":0}],"content":"Daron Acemoglu\nDepartment of Economics, E52-446\nMassachusetts Institute of Technology\n77 Massachusetts Avenue\nCambridge, MA 02139\nand NBER\ndaron@mit.edu","description":null,"id":3,"type":"text"},{"bbox":[{"coord":[117,860,430,1021],"page_id":0}],"content":"Asuman Ozdaglar\nDepartment of Electrical Engineering\nand Computer Science\nMassachusetts Institute of Technology\n77 Massachusetts Ave, E40-130\nCambridge, MA 02139\nasuman@mit.edu","description":null,"id":4,"type":"text"},{"bbox":[{"coord":[508,673,821,834],"page_id":0}],"content":"James Siderius\nDepartment of Electrical Engineering\nand Computer Science\nMassachusetts Institute of Technology\n77 Massachusetts Ave.\nCambridge, MA 02139\nsiderius@mit.edu","description":null,"id":5,"type":"text"}],"pages":[{"id":0,"image_uri":null}]},"error_status":null,"metadata":{"file_metadata":null,"id":"993dc9c3-39f9-4fb8-92e1-a507e956d819","version":"2.0"}} dbfs:/Volumes/studies/databricks_finetuning/article_infos/pgs19_20_regulation.pdf {"document":{"elements":[{"bbox":[{"coord":[94,94,935,203],"page_id":0}],"content":"Remark — In Theorem 3, we assume the platform can select any network P it desires through its recommendation algorithm. This is without loss of generality. If we assume the social network originally begins as an arbitrary island network in Section 4, and the platform can hide and amplify content across different links, the same result readily follows.","description":null,"id":0,"type":"text"},{"bbox":[{"coord":[96,236,675,259],"page_id":0}],"content":"5.2 Comparative Statics on $\\gamma_p$ : Divisiveness and Polarization","description":null,"id":1,"type":"section_header"},{"bbox":[{"coord":[96,278,935,357],"page_id":0}],"content":"In Theorem 3, the threshold $r_p$ fully summarizes the extent to which misinformation will spread virally on social media. We next perform comparative statics for this threshold to understand the conditions under which the platform will create a filter bubble and propagate misinformation.","description":null,"id":2,"type":"text"},{"bbox":[{"coord":[96,371,

In [0]:
if RUN_PARSE_STEP:
    # export as table
    output_table_name = 'article_parsed_infos'
    parsed_docs_sql.write.format('delta').mode('overwrite').saveAsTable(output_table_name)
else:
    pass

<img src="./imgs/table_article_parsed_infos.png">

## Processing parsed data

In [0]:
assert ws_client.vector_search_endpoints.create_endpoint != None

In [0]:
if RUN_CHUNK_STEP:
    plain_text = parsed_docs_sql.withColumn(
        'plain_text',
        expr('''
            concat_ws(
                "\n",
                transform(
                    from_json(
                        to_json(parsed_docs:document.elements),
                        'array<struct<
                            bbox:array<struct<coord:array<int>,page_id:int>>,
                            content:string,
                            description:string,
                            id:int,
                            type:string
                        >>'
                    ),
                    x -> x.content
                )
            )
        ''')
    )

    display(plain_text.select('path', 'plain_text'))

In [0]:
if RUN_CHUNK_STEP:
	chunk_size = 300
	chunk_overlap = 30

	text_splitter = RecursiveCharacterTextSplitter(
		chunk_size=chunk_size,
		chunk_overlap=chunk_overlap,
		separators=['\n\n', '\n', '', ' ']
	)

	chunk_schema = StructType([
		StructField('path', StringType(), True),
		StructField('chunk', StringType(), True),
	])


	def split_rows(iterator):
		for pdf in iterator:
			out = []
			for _, row in pdf.iterrows():
				path = row['path']
				text = row['plain_text']
				if isinstance(text, str) and text.strip():
					for c in text_splitter.split_text(text):
						if c and c.strip():
							out.append((path, c))
			yield pd.DataFrame(out, columns=['path', 'chunk'])
	

	# chunking
	df_chunks = (
		plain_text
		.select('path', 'plain_text')
		.mapInPandas(split_rows, schema=chunk_schema)
	)
else:
    df_chunks = spark.read.table(f'{CATALOG}.{SCHEMA}.{CHUNKS_TABLE_NAME}')

display(df_chunks)

path,chunk,id
dbfs:/Volumes/studies/databricks_finetuning/article_infos/pg2_abstract.pdf,"A Model of Online Misinformation Daron Acemoglu, Asuman Ozdaglar, and James Siderius NBER Working Paper No. 28884 June 2021, Revised January 2022 JEL No. D83,D85,P16 ABSTRACT",0
dbfs:/Volumes/studies/databricks_finetuning/article_infos/pg2_abstract.pdf,We present a model of online content sharing where agents sequentially observe an article and must decide whether to share it with others. This content may or may not contain misinformation. Agents gain utility from positive social media interactions but do not want to be called out for propagating,1
dbfs:/Volumes/studies/databricks_finetuning/article_infos/pg2_abstract.pdf,be called out for propagating misinformation. We characterize the (Bayesian-Nash) equilibria of this social media game and show sharing exhibits strategic complementarity. Our first main result establishes that the impact of homophily on content virality is non-monotone: homophily reduces the broad,2
dbfs:/Volumes/studies/databricks_finetuning/article_infos/pg2_abstract.pdf,"e: homophily reduces the broader circulation of an article, but it creates echo chambers that impose less discipline on the sharing of low-reliability content. This insight underpins our second main result, which demonstrates that social media platforms interested in maximizing engagement tend to de",3
dbfs:/Volumes/studies/databricks_finetuning/article_infos/pg2_abstract.pdf,"ximizing engagement tend to design their algorithms to create more homophilic communication patterns (""filter bubbles""). We show that platform incentives to amplify misinformation are particularly pronounced for low-reliability content likely to contain misinformation and when there is greater polar",4
dbfs:/Volumes/studies/databricks_finetuning/article_infos/pg2_abstract.pdf,"nd when there is greater polarization and more divisive content. Finally, we discuss various regulatory solutions to such platform-manufactured misinformation.",5
dbfs:/Volumes/studies/databricks_finetuning/article_infos/pg2_abstract.pdf,"Daron Acemoglu Department of Economics, E52-446 Massachusetts Institute of Technology 77 Massachusetts Avenue Cambridge, MA 02139 and NBER daron@mit.edu Asuman Ozdaglar Department of Electrical Engineering and Computer Science Massachusetts Institute of Technology 77 Massachusetts Ave, E40-130",6
dbfs:/Volumes/studies/databricks_finetuning/article_infos/pg2_abstract.pdf,"77 Massachusetts Ave, E40-130 Cambridge, MA 02139 asuman@mit.edu James Siderius Department of Electrical Engineering and Computer Science Massachusetts Institute of Technology 77 Massachusetts Ave. Cambridge, MA 02139 siderius@mit.edu",7
dbfs:/Volumes/studies/databricks_finetuning/article_infos/pgs19_20_regulation.pdf,"Remark — In Theorem 3, we assume the platform can select any network P it desires through its recommendation algorithm. This is without loss of generality. If we assume the social network originally begins as an arbitrary island network in Section 4, and the platform can hide and amplify content acr",8
dbfs:/Volumes/studies/databricks_finetuning/article_infos/pgs19_20_regulation.pdf,"n hide and amplify content across different links, the same result readily follows.",9


In [0]:
if RUN_CHUNK_STEP:
    # export chunks table
    df_chunks_for_embedding = df_chunks.withColumn('id', monotonically_increasing_id())
    df_chunks_for_embedding.write.format('delta').mode('overwrite').option('mergeSchema', 'true').saveAsTable('article_chunks')
else:
    pass

<img src='./imgs/table_article_chunks.png'>

## Create vs index

In [0]:
def create_vs_index(vs_index_name: str, vs_enpoint_name: str, vs_client: WorkspaceClient) -> None:
	'''Create a vector search index for a endpoint.'''
 
	# enable cdf (change data feed)
	spark.sql(f'''alter table {CHUNKS_TABLE_NAME} set tblproperties (delta.enableChangeDataFeed = true)''')

	source_table_name = f'{CATALOG}.{SCHEMA}.{CHUNKS_TABLE_NAME}'
 
	vs_client.create_delta_sync_index_and_wait(
		endpoint_name=vs_enpoint_name,
		index_name=f'{CATALOG}.{SCHEMA}.{vs_index_name}',
		primary_key='id',
		source_table_name=source_table_name,
		pipeline_type='TRIGGERED',
		embedding_source_column='chunk',
		embedding_model_endpoint_name='databricks-gte-large-en',
		verbose=True
	)

	print(f'Source table: {source_table_name}')
	print(f'Vector store index: {vs_enpoint_name}')
	print(f'Endpoint: {vs_enpoint_name}')

In [0]:
for endpoint in ws_client.vector_search_endpoints.list_endpoints():
    print(endpoint.name)

article_vs_endpoint


In [0]:
if CREATE_VS_INDEX:
    create_vs_index(
        vs_index_name=VS_INDEX_NAME,
        vs_enpoint_name=VS_ENDPOINT_NAME,
        vs_client=vs_client
        )

In [0]:
indexes = list(
    ws_client.vector_search_indexes.list_indexes('article_vs_endpoint')
)
print([index.name for index in indexes])

['studies.databricks_finetuning.article_vs_index']


<img src='./imgs/article_vs_index.png'>

## Query the vs endpoint

In [0]:
def query_vs_docs(
  query_text: str,
  index_name: str, 
  num_results: int,
  vs_client: VectorSearchClient
) -> List[dict]:
	'''Search on vector index and returns results.'''
	
	cols = ['path', 'chunk', 'id']
	res = vs_client.get_index(index_name=index_name).similarity_search(
		query_text=query_text,
		num_results=num_results,
		columns=cols
	)
	res_pd = pd.DataFrame(res['result']['data_array'])
	res_pd.columns = [col['name'] for col in res['manifest']['columns']]
	return json.loads(res_pd.to_json(orient='records'))

In [0]:
question = '''What were the principal results of the paper?'''
ans = query_vs_docs(
        query_text=question,
        index_name= f'{CATALOG}.{SCHEMA}.{VS_INDEX_NAME}',
        num_results=2,
        vs_client=vs_client
    )

print(ans)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[{'path': 'dbfs:/Volumes/studies/databricks_finetuning/article_infos/pgs25_6_conclusion.pdf', 'chunk': '7 Conclusion', 'id': 40.0, 'score': 0.5823039859}, {'path': 'dbfs:/Volumes/studies/databricks_finetuning/article_infos/pgs3_6_intro.pdf', 'chunk': 'The rest of the paper is organized as follows. The next section introduces our basic environment and describes the information structure and payoffs. Section 3 characterizes the (Bayesian-Nash) equilibria of this model and provides some basic comparative static results. Section 4 studies the effect', 'id': 186.0, 'score': 0.5489477631}]


In [0]:
# Suggested questions #

# question = '''What were the principal results of the paper?'''
# question = '''What was the proposed model?'''
# question = '''Are there any weaknesses in the presented hypothesis?'''
# question = '''What is the impact of the article's results?'''
# question = '''What are the limitations of the presented model?'''
# question = '''Who are the references most cited by the authors?'''
# question = '''What are the direct connections of the work?'''
# question = '''How does the work propose to treat misinformation?'''

In [0]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

while 1 > 0:
	question = input('Question: ')
	if len(question) == 0:
		break 
	else:
		ans = query_vs_docs(
			query_text=question,
			index_name= f'{CATALOG}.{SCHEMA}.{VS_INDEX_NAME}',
			num_results=2,
			vs_client=vs_client
		)

		display(pd.DataFrame(ans))
		print(40*'-')

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 50)

Question:  What were the principal results of the paper?

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


path,chunk,id,score
dbfs:/Volumes/studies/databricks_finetuning/article_infos/pgs25_6_conclusion.pdf,7 Conclusion,40.0,0.5823039859
dbfs:/Volumes/studies/databricks_finetuning/article_infos/pgs3_6_intro.pdf,The rest of the paper is organized as follows. The next section introduces our basic environment and describes the information structure and payoffs. Section 3 characterizes the (Bayesian-Nash) equilibria of this model and provides some basic comparative static results. Section 4 studies the effect,186.0,0.5489477631


----------------------------------------


Question:  What was the proposed model?

...

Question:  How does the work propose to treat misinformation?

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


path,chunk,id,score
dbfs:/Volumes/studies/databricks_finetuning/article_infos/pgs3_6_intro.pdf,"show how they may reduce misinformation but also point out the possibility that, if they are not designed well, they can backfire and exacerbate the problem.",164.0,0.6561323398
dbfs:/Volumes/studies/databricks_finetuning/article_infos/pgs7_9_model.pdf,"Our focus in this paper is on misinformation, interpreted as items containing misleading information or arguments that can influence (a subset of) the public. Articles containing misinformation are in practice much more numerous than those that can be classified as ""fake news"", which explicitly pro",207.0,0.6410317885


----------------------------------------


Question:  